# Prompt Evaluation (Đánh giá Prompt)

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

### Tại sao cần đánh giá prompt?
Khi sửa prompt, làm sao biết nó **tốt hơn hay tệ hơn**? Nhìn 1-2 ví dụ thì cảm tính, không đáng tin. Giải pháp: chạy prompt trên **nhiều test case**, cho ra **điểm số** → so sánh khách quan.

> 💡 *"You can't improve what you can't measure"*

### Quy trình 4 bước
```
Dataset  →  Run Prompt  →  Grade  →  Aggregate
(dữ liệu)   (chạy prompt)  (chấm)   (tổng hợp điểm)
```


In [ ]:
!pip install anthropic

In [ ]:
import json
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

## Bước 1: Tạo bộ dữ liệu test (Dataset)

Mỗi test case mô tả 1 **nhiệm vụ (`task`)** + **tiêu chí chấm (`solution_criteria`)**. Có thể viết tay (chính xác) hoặc nhờ Claude tự sinh (nhanh, đa dạng).


In [ ]:
dataset = [
    {
        "task": "Viết hàm Python tính giai thừa của một số nguyên dương.",
        "solution_criteria": "Phải xử lý đúng trường hợp n=0 (trả về 1) "
                             "và dùng vòng lặp hoặc đệ quy chính xác.",
    },
    {
        "task": "Viết hàm Python kiểm tra một chuỗi có phải palindrome không.",
        "solution_criteria": "Phải không phân biệt hoa thường và xử lý chuỗi rỗng.",
    },
    {
        "task": "Viết hàm Python tìm số lớn nhất trong một danh sách.",
        "solution_criteria": "Phải xử lý danh sách rỗng một cách an toàn.",
    },
]

### (Tùy chọn) Nhờ Claude tự sinh dataset

Dùng **structured output** (`output_config`) để ép Claude trả về đúng JSON.


In [ ]:
def generate_dataset():
    prompt = """
Hãy tạo 3 nhiệm vụ lập trình Python ở mức cơ bản để kiểm tra một AI coding assistant.
Mỗi nhiệm vụ gồm:
- "task": mô tả nhiệm vụ
- "solution_criteria": tiêu chí để đánh giá lời giải đúng
""".strip()

    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        messages=[{"role": "user", "content": prompt}],
        output_config={
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {
                        "tasks": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "task": {"type": "string"},
                                    "solution_criteria": {"type": "string"},
                                },
                                "required": ["task", "solution_criteria"],
                                "additionalProperties": False,
                            },
                        }
                    },
                    "required": ["tasks"],
                    "additionalProperties": False,
                },
            }
        },
    )
    return json.loads(response.content[0].text)["tasks"]

# generate_dataset()   # bỏ comment để thử

## Bước 2: Chạy prompt cần đánh giá

Đây là prompt mà ta muốn **ĐO chất lượng**. Hàm nhận 1 test case, ghép vào prompt, gọi Claude, trả về output.


In [ ]:
def run_prompt(test_case):
    # ===== ĐÂY LÀ PROMPT BẠN MUỐN ĐÁNH GIÁ / CẢI THIỆN =====
    prompt = f"""
Hãy giải nhiệm vụ lập trình sau bằng Python.
Chỉ trả về code, kèm comment ngắn gọn giải thích.

Nhiệm vụ: {test_case["task"]}
""".strip()

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

## Bước 3: Chấm điểm (Grading) — 3 phương pháp

| Phương pháp | Khi nào dùng | Ưu / Nhược |
|---|---|---|
| **Code-based** | Đáp án rõ ràng (khớp, chứa từ khóa, chạy code) | ✅ Nhanh, rẻ / ❌ Chỉ hợp bài đáp án cứng |
| **Model-based** (LLM as Judge) | Đáp án mở, cần đánh giá chất lượng | ✅ Linh hoạt / ❌ Tốn token |
| **Human** | Task chủ quan | ✅ Chính xác nhất / ❌ Chậm, đắt |


### 3a. Code-based grading
Dùng luật cứng để chấm. Ví dụ: kiểm tra output có chứa `def ` không.


In [ ]:
def grade_by_code(output):
    score = 10 if "def " in output else 0
    return {"score": score,
            "reasoning": "Có chứa định nghĩa hàm" if score else "Thiếu hàm"}

### 3b. Model-based grading — "LLM as a Judge" ⭐

Dùng **một con Claude khác làm giám khảo** chấm output. **Mẹo quan trọng:** yêu cầu `reasoning` (lý do) **trước**, `score` (điểm) **sau** → bắt model suy nghĩ trước khi cho điểm → điểm chính xác & nhất quán hơn.


In [ ]:
def grade_by_model(test_case, output):
    grader_prompt = f"""
Bạn là một giám khảo nghiêm khắc đánh giá lời giải lập trình.

<nhiem_vu>
{test_case["task"]}
</nhiem_vu>

<tieu_chi_danh_gia>
{test_case["solution_criteria"]}
</tieu_chi_danh_gia>

<loi_giai_can_cham>
{output}
</loi_giai_can_cham>

Hãy chấm điểm lời giải từ 1 đến 10 dựa trên tính đúng đắn, đầy đủ
và việc đáp ứng tiêu chí. Giải thích lý do ngắn gọn.
""".strip()

    response = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": grader_prompt}],
        output_config={
            "format": {
                "type": "json_schema",
                "schema": {
                    "type": "object",
                    "properties": {
                        "reasoning": {"type": "string"},
                        "score": {"type": "integer"},
                    },
                    "required": ["reasoning", "score"],   # reasoning TRƯỚC score
                    "additionalProperties": False,
                },
            }
        },
    )
    return json.loads(response.content[0].text)

### 3c. Human grading
Con người chấm — chính xác nhất cho task chủ quan, nhưng chậm và tốn kém. Thường dùng để tạo "chuẩn vàng" hiệu chỉnh lại model-grader. (Không có code mẫu.)


## Bước 4: Ghép tất cả — chạy eval trên cả dataset

Tổng hợp cuối cùng là **điểm trung bình** — con số dùng để so sánh giữa các phiên bản prompt.


In [ ]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    grade = grade_by_model(test_case, output)
    return {
        "task": test_case["task"],
        "output": output,
        "score": grade["score"],
        "reasoning": grade["reasoning"],
    }

def run_eval(dataset):
    results = []
    for i, test_case in enumerate(dataset, 1):
        print(f"\n=== Test {i}/{len(dataset)} ===")
        result = run_test_case(test_case)
        results.append(result)
        print(f"Nhiệm vụ : {result['task']}")
        print(f"Điểm     : {result['score']}/10")
        print(f"Lý do    : {result['reasoning']}")

    avg = sum(r["score"] for r in results) / len(results)
    print(f"\n{'='*50}")
    print(f"ĐIỂM TRUNG BÌNH: {avg:.2f}/10")
    print(f"{'='*50}")
    return results

## Chạy thử

In [ ]:
results = run_eval(dataset)

---
## ✅ Tóm tắt

- **Đánh giá prompt = đo lường khách quan** thay vì cảm tính.
- Quy trình: Dataset → Run → Grade → Aggregate.
- 3 cách chấm: code-based, model-based (LLM judge), human.
- **LLM as a Judge** là kỹ thuật chủ lực: nhớ cho `reasoning` trước `score`.

> ⚠️ Khác khóa cũ: dùng `output_config` (không prefill `{`) để lấy JSON — prefill đã bị bỏ trên Opus 4.8.
